# DeepSeek-OCR Backend Server (untuk GMP Automation System)

Notebook ini menjalankan model [deepseek-ai/DeepSeek-OCR](https://huggingface.co/deepseek-ai/DeepSeek-OCR) di Kaggle (GPU) dan meng-expose-nya sebagai HTTP API lewat ngrok tunnel, supaya aplikasi lokal (`gmp_automation`) bisa memanggilnya menggantikan Claude API.

**Sebelum run:**
1. Di kanan notebook: Settings → Accelerator → pilih **GPU T4 x2** (atau P100).
2. Settings → Internet → **ON** (wajib, untuk download model & ngrok tunnel).
3. Tambahkan ngrok authtoken sebagai Kaggle Secret bernama `NGROK_AUTHTOKEN`.
   - Daftar gratis di https://dashboard.ngrok.com/get-started/your-authtoken
   - Add-ons → Secrets → Add Secret → Label: `NGROK_AUTHTOKEN`, Value: token Anda.
4. Run All. URL publik akan muncul di output cell terakhir — copy URL itu (format `https://xxxx.ngrok-free.app`) dan tempel ke aplikasi lokal (config.py / UI).

In [ ]:
# 1. Install dependencies
!pip install -q -U transformers==4.46.3 tokenizers==0.20.3 einops addict easydict
!pip install -q -U fastapi uvicorn pyngrok python-multipart nest_asyncio
# flash-attn is optional (speeds up inference); skip if it fails to build
!pip install -q flash-attn==2.7.3 --no-build-isolation || echo 'flash-attn install skipped, will run without it'

In [ ]:
# 2. Load the DeepSeek-OCR model
import torch
from transformers import AutoModel, AutoTokenizer

MODEL_NAME = 'deepseek-ai/DeepSeek-OCR'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

try:
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        _attn_implementation='flash_attention_2',
        trust_remote_code=True,
        use_safetensors=True,
    )
except Exception as e:
    print('flash_attention_2 unavailable, falling back to eager attention:', e)
    model = AutoModel.from_pretrained(
        MODEL_NAME,
        _attn_implementation='eager',
        trust_remote_code=True,
        use_safetensors=True,
    )

model = model.eval().cuda().to(torch.bfloat16)
print('Model loaded.')

In [ ]:
# 3. FastAPI server exposing /ocr
import base64
import os
import shutil
import tempfile
import uuid

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI(title='DeepSeek-OCR GMP Backend')

# Prompt used for document -> markdown conversion (see HF model card).
DOC_TO_MARKDOWN_PROMPT = "<image>\n<|grounding|>Convert the document to markdown. "
FREE_OCR_PROMPT = "<image>\nFree OCR. "


class OcrRequest(BaseModel):
    image_b64: str          # PNG image, base64-encoded (no data: prefix)
    mode: str = 'markdown'  # 'markdown' or 'free'
    resolution: str = 'gundam'  # tiny/small/base/large/gundam


RESOLUTION_PRESETS = {
    'tiny':   dict(base_size=512,  image_size=512, crop_mode=False),
    'small':  dict(base_size=640,  image_size=640, crop_mode=False),
    'base':   dict(base_size=1024, image_size=1024, crop_mode=False),
    'large':  dict(base_size=1280, image_size=1280, crop_mode=False),
    'gundam': dict(base_size=1024, image_size=640, crop_mode=True),
}


@app.get('/health')
def health():
    return {'status': 'ok', 'model': MODEL_NAME}


@app.post('/ocr')
def ocr(req: OcrRequest):
    if req.resolution not in RESOLUTION_PRESETS:
        raise HTTPException(400, f'Unknown resolution preset: {req.resolution}')

    prompt = DOC_TO_MARKDOWN_PROMPT if req.mode == 'markdown' else FREE_OCR_PROMPT
    preset = RESOLUTION_PRESETS[req.resolution]

    work_dir = tempfile.mkdtemp(prefix='dsocr_')
    image_path = os.path.join(work_dir, f'{uuid.uuid4().hex}.png')
    try:
        with open(image_path, 'wb') as f:
            f.write(base64.b64decode(req.image_b64))

        result_text = model.infer(
            tokenizer,
            prompt=prompt,
            image_file=image_path,
            output_path=work_dir,
            base_size=preset['base_size'],
            image_size=preset['image_size'],
            crop_mode=preset['crop_mode'],
            save_results=False,
        )

        # model.infer() in some versions returns text directly; in others it
        # writes a result.mmd/result.md file into output_path instead.
        if not result_text or not isinstance(result_text, str):
            for fname in ('result.mmd', 'result.md', 'result.txt'):
                fpath = os.path.join(work_dir, fname)
                if os.path.exists(fpath):
                    with open(fpath, 'r', encoding='utf-8') as rf:
                        result_text = rf.read()
                    break

        return {'text': result_text or ''}
    finally:
        shutil.rmtree(work_dir, ignore_errors=True)

In [ ]:
# 4. Start server + ngrok tunnel
import nest_asyncio
import uvicorn
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

nest_asyncio.apply()

authtoken = UserSecretsClient().get_secret('NGROK_AUTHTOKEN')
conf.get_default().auth_token = authtoken

public_url = ngrok.connect(8000, 'http')
print('=' * 70)
print(f'DeepSeek-OCR backend is live at: {public_url}')
print('Copy this URL into the GMP Automation app (DeepSeek-OCR endpoint field).')
print('=' * 70)

uvicorn.run(app, host='0.0.0.0', port=8000)